[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Wahyudi-Satriawan/244107020137_TI3H_PembelajaranMesin/blob/main/Jobsheet%20Dataset%20PBL/JS02.ipynb)

# JOBSHEET 02: PEMAHAMAN DATA (EDA) DAN PRA-PENGOLAHAN DATA
### Eksplorasi, Audit Anomali, dan Pembersihan Empiris pada Dataset Mentah Properti PBL RumahPoint
**Mata Kuliah:** Pembelajaran Mesin — Semester 5 (Tahun Akademik 2025/2026)  
**Mahasiswa:** Wahyudi Satriawan Hamid  
**NIM / Kelas:** 244107020137 / TI-3H  
**Program Studi:** D-IV Teknik Informatika — Jurusan Teknologi Informasi — Politeknik Negeri Malang  

---

## Ringkasan Eksekutif & Peta Adaptasi Modul ke Dataset PBL
Jobsheet ini berfokus pada penguasaan tahapan fundamental rekayasa data machine learning: **Exploratory Data Analysis (EDA)**, deteksi anomali/outlier, penanganan missing values, encoding variabel kategorikal, dan standardisasi fitur numerik.

Sesuai kontrak teknis pada `DECISION_LOG.md`, jobsheet ini menerapkan seluruh konsep inti modul Polinema langsung ke dataset riil berskala industri (**Dataset Mentah Properti Multi Kota Indonesia**, 64.650 baris data mentah):

| Bagian Modul | Topik & Metode Asli Modul | Adaptasi & Implementasi Dataset PBL | Justifikasi Metodologis & Anti-Leakage |
|:---|:---|:---|:---|
| **Bagian 1** | **Praktikum 1:** EDA & Inspeksi Data (Dataset Titanic) | Audit menyeluruh data mentah `properti_multi_kota.csv` (64.650 baris, 8 fitur): dimensi, tipe data, ringkasan statistik (P1–P99), duplikasi, missing values, kardinalitas, dan visualisasi bar chart & boxplot log-scale. | Memberikan gambaran empiris riil atas noise data hasil web scraping sebelum proses pembersihan dijalankan. |
| **Bagian 2** | **Praktikum 2:** Pembersihan Data Terstruktur & Imputasi Dasar | Pembersihan terverifikasi empiris 5 tahap: (1) Deduplikasi persis, (2) Filter outlier domain, (3) Perbaikan usia bangunan & ekstraksi kondisi, (4) Normalisasi lokasi, (5) Pembentukan target seimbang `kelas_harga` & `kelas_harga_biner`. | Memastikan integritas data tanpa kebocoran (*data leakage*). Retensi sampel komersial (`jumlah_kamar == 0`) dibuktikan secara ekonomi (median Rp 1,34 M). Menghasilkan tepat 56.170 baris. |
| **Bagian 3** | **Praktikum 3 & Tugas:** Seleksi Fitur, Encoding, & Standardisasi (Titanic & WBC) | Demonstrasi eksploratif imputasi median, komparasi Frequency Encoding vs OHE Top-15 + 'Lainnya', serta StandardScaler numerik & transformasi logaritmik target `log1p(harga)`. | **Anti-Leakage Showcase:** Penjelasan eksplisit pemisahan antara demonstrasi eksploratif di JS02 vs implementasi produksi di JS04–JS15 di mana transformasi wajib masuk ke Pipeline setelah split. |
| **Bagian 4** | **Inspeksi Korelasi & Eksplorasi Pasca-Praproses** | Visualisasi 3 panel (boxplot harga vs kondisi, boxplot luas vs kelas harga, scatter luas vs harga log-scale) dan heatmap korelasi fitur numerik terhadap harga & `log1p(harga)`. | Memberikan insight domain bisnis properti untuk perancangan fitur prediktif pada modul lanjutan. |
| **Pengecualian Sah** | **Praktikum 4:** Pra-pengolahan Data Citra (`lena.jpg`) | Dikecualikan pada versi PBL: materi citra dipertahankan penuh pada `Jobsheet Sesuai Modul/JS02.ipynb` (Versi 1). Dataset PBL murni tabular relasional. | Ketiadaan file citra di `Dataset/` dan fokus kompetensi penanganan data terstruktur skala besar. |

## Justifikasi Ilmiah & Metodologis Adaptasi Modul

### 1. Fokus Penuh pada Dataset Mentah Properti PBL (`properti_multi_kota.csv`)
Pada buku modul asli Polinema, materi `JS02` dipraktikkan menggunakan dataset bawaan Titanic (891 baris) dan Wisconsin Breast Cancer (569 baris). Pada repositori ini, kedua praktikum standar tersebut telah diselesaikan dan diarsipkan secara utuh pada `Jobsheet Sesuai Modul/JS02.ipynb` (Versi 1).

Untuk repositori versi PBL (`Jobsheet Dataset PBL/JS02.ipynb`), fokus dialihkan 100% pada dataset riil proyek PBL: **`Dataset/properti_multi_kota.csv`** yang memiliki **64.650 baris mentah** dan 8 atribut heterogen. Dataset ini bersumber dari scraping multi-portal properti daring di Indonesia, sehingga menghadirkan tantangan riil dunia data:
1. **Duplikasi persis** akibat scraping multi-portal terhadap listing properti yang sama.
2. **Outlier fisik dan finansial ekstrem** (misalnya harga Rp 1.000 atau tanah seluas 1 m²).
3. **Format tanggal tidak seragam** (tahun kalender bercampur angka usia relatif dan placeholder scraper).
4. **Distribusi harga berskewness positif ekstrem (*heavy right-skewed*)** yang membutuhkan penanganan khusus.

### 2. Pengecualian Metodologis Praktikum Citra (`Praktikum 4 — lena.jpg`)
Pada sub-praktikum 4 modul Polinema, diperkenalkan manipulasi piksel citra dasar menggunakan OpenCV (`cv2.resize` dan `cv2.cvtColor(BGR2GRAY)` pada `lena.jpg`). Sub-praktikum ini **secara sadar dikecualikan** pada versi PBL karena dua alasan:
- **Ketiadaan Data Citra:** Folder `Dataset/` hanya berisi data tabular relasional (`properti_multi_kota.csv`), tanpa adanya data citra foto properti.
- **Konsistensi Domain:** Jobsheet 02 versi PBL didesain sebagai fondasi pra-pengolahan tabular untuk seluruh rantai pemodelan machine learning (`JS04` hingga `JS15`). Sub-praktikum citra sudah terselesaikan secara lengkap di Versi 1.

In [ ]:
# ==============================================================================
# PENGATURAN AWAL: IMPORT LIBRARY, SEED REPRODUSIBILITAS, & KONFIGURASI VISUAL
# ==============================================================================
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn

# Scikit-Learn Preprocessing & Imputation
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

# Kunci random seed untuk reprodusibilitas
np.random.seed(42)

# Konfigurasi visualisasi
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.autolayout'] = True
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'

print("=" * 75)
print("SISTEM INFORMASI LINGKUNGAN KOMPUTASI (GOOGLE COLAB / LOCAL JUPYTER)")
print("=" * 75)
print(f"Versi Python       : {sys.version.split()[0]}")
print(f"Versi NumPy        : {np.__version__}")
print(f"Versi Pandas       : {pd.__version__}")
print(f"Versi Scikit-Learn : {sklearn.__version__}")
print("=" * 75)

---
## Bagian 1 — Pemuatan Data Mentah & Audit EDA Awal (Adaptasi Praktikum 1)

Pada bagian ini, kita menerapkan tahapan inspeksi data eksploratif (EDA) seperti pada Praktikum 1 modul Polinema, meliputi:
1. Pemuatan data melalui **Dual-Mode Loader** (otomatis mendeteksi lingkungan lokal vs Google Colab).
2. Pengecekan dimensi, struktur baris-kolom, dan tipe data.
3. Statistik deskriptif komprehensif dengan cakupan persentil ekstrem (P1 hingga P99).
4. Audit duplikasi baris dan kardinalitas fitur kategorikal.
5. Audit menyeluruh terhadap nilai yang hilang (*missing values*).
6. Deteksi nilai anomali dan pelanggaran batas domain fisik properti.
7. Visualisasi awal profil missing values dan sebaran outlier ekstrem.

In [ ]:
# ==============================================================================
# LOADER DUAL-MODE: PEMUATAN DATASET MENTAH (PROPERTI_MULTI_KOTA.CSV)
# ==============================================================================
DATASET_FILENAME = "properti_multi_kota.csv"
LOCAL_PATH = os.path.join("..", "Dataset", DATASET_FILENAME)
GITHUB_RAW_URL = f"https://raw.githubusercontent.com/Wahyudi-Satriawan/244107020137_TI3H_PembelajaranMesin/main/Dataset/{DATASET_FILENAME}"

if os.path.exists(LOCAL_PATH):
    print(f"[LOADER] Memuat dataset mentah dari path lokal: {LOCAL_PATH}")
    df_raw = pd.read_csv(LOCAL_PATH)
else:
    print(f"[LOADER] Path lokal tidak ditemukan. Mengunduh dataset mentah dari GitHub RAW: {GITHUB_RAW_URL}")
    df_raw = pd.read_csv(GITHUB_RAW_URL)

print(f"[LOADER] Status: Berhasil memuat {df_raw.shape[0]:,} baris dan {df_raw.shape[1]} kolom.")

### 1.1 Audit Dimensi, Struktur Kolom, dan Tipe Data
Memeriksa ukuran matriks dataset mentah, nama seluruh variabel, dan tipe representasi data di memori.

In [ ]:
print("=" * 60)
print("AUDIT DIMENSI DAN STRUKTUR KOLOM DATASET MENTAH")
print("=" * 60)
print(f"Jumlah Baris : {df_raw.shape[0]:,}")
print(f"Jumlah Kolom : {df_raw.shape[1]}")
print("\nStruktur Kolom dan Tipe Data:")
print(df_raw.dtypes)
print("\nRingkasan Informasi DataFrame (df_raw.info()):")
df_raw.info()
print("\nCuplikan 5 Baris Pertama:")
display(df_raw.head())

### 1.2 Ringkasan Statistik Deskriptif & Deteksi Persentil Ekstrem
Memeriksa sebaran statistik fitur numerik dengan mengevaluasi persentil 1%, 5%, 25%, 50%, 75%, 95%, dan 99% guna mengidentifikasi *heavy right-skewed tail* dan anomali nilai minimum.

In [ ]:
percentiles = [0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]
desc_stats = df_raw.describe(percentiles=percentiles).T
print("=" * 80)
print("STATISTIK DESKRIPTIF FITUR NUMERIK (DENGAN PERSENTIL EKSTREM P1 - P99)")
print("=" * 80)
display(desc_stats.style.format("{:,.2f}"))

### 1.3 Audit Duplikasi Data dan Kardinalitas Fitur Kategorikal
Mendeteksi keberadaan baris identik (duplikat persis) dan mengevaluasi keragaman kategori pada kolom teks (`lokasi` dan `sumber`).

In [ ]:
n_duplicates = df_raw.duplicated().sum()
pct_duplicates = (n_duplicates / len(df_raw)) * 100

print("=" * 60)
print("AUDIT DUPLIKASI DATA MENTAH")
print("=" * 60)
print(f"Jumlah Baris Duplikat Persis : {n_duplicates:,} baris ({pct_duplicates:.2f}%)")
print(f"Jumlah Baris Unik             : {len(df_raw) - n_duplicates:,} baris ({100 - pct_duplicates:.2f}%)")

print("\n" + "=" * 60)
print("AUDIT KARDINALITAS KOLOM KATEGORIKAL")
print("=" * 60)
print(f"Jumlah Nilai Unik Kolom 'lokasi' : {df_raw['lokasi'].nunique():,} kota/kabupaten")
print(f"Jumlah Nilai Unik Kolom 'sumber' : {df_raw['sumber'].nunique():,} sumber portal")
print("\nDistribusi Frekuensi Sumber Data:")
print(df_raw['sumber'].value_counts(dropna=False))

### 1.4 Audit Nilai Hilang (*Missing Values*) per Kolom
Mengidentifikasi tingkat kekosongan data per atribut untuk menentukan kelayakan kolom sebagai fitur pemodelan.

In [ ]:
missing_count = df_raw.isnull().sum()
missing_pct = (df_raw.isnull().sum() / len(df_raw)) * 100
missing_df = pd.DataFrame({
    'Jumlah_Missing': missing_count,
    'Persentase_Missing (%)': missing_pct,
    'Tipe_Data': df_raw.dtypes
}).sort_values(by='Jumlah_Missing', ascending=False)

print("=" * 70)
print("AUDIT NILAI HILANG (MISSING VALUES) PER KOLOM")
print("=" * 70)
display(missing_df.style.format({'Persentase_Missing (%)': '{:.2f}%', 'Jumlah_Missing': '{:,}'}))

### 1.5 Audit Nilai Anomali dan Batas Domain Fisik Properti
Melakukan audit empiris terhadap ambang batas domain fisik properti di Indonesia:
- **Harga:** Mengidentifikasi listing dengan harga di bawah Rp 50 Juta (biasanya harga sewa bulanan atau placeholder) atau di atas Rp 100 Miliar (listing anomali/typo scraper).
- **Luas Tanah (`luas_m2`):** Mengidentifikasi luas di bawah 10 m² atau di atas 2.000 m².
- **Jumlah Kamar Tidur & Kamar Mandi:** Mengidentifikasi listing dengan kamar > 20, serta menganalisis fenomena listing `jumlah_kamar == 0`.
- **Usia Bangunan:** Mengidentifikasi placeholder scraper berupa angka 3–4 digit (rentang 77..1899) yang bukan tahun kalender valid maupun umur bangunan logis.

In [ ]:
print("=" * 75)
print("AUDIT NILAI ANOMALI DAN BATAS DOMAIN FISIK PROPERTI")
print("=" * 75)

# 1. Anomali Harga
harga_bawah = (df_raw['harga'] < 50_000_000).sum()
harga_atas = (df_raw['harga'] > 100_000_000_000).sum()
print(f"1. Anomali Harga (< Rp 50 Jt): {harga_bawah:,} baris | (> Rp 100 Miliar): {harga_atas:,} baris")

# 2. Anomali Luas Tanah (luas_m2)
luas_bawah = (df_raw['luas_m2'] < 10).sum()
luas_atas = (df_raw['luas_m2'] > 2000).sum()
print(f"2. Anomali Luas Tanah (< 10 m2): {luas_bawah:,} baris | (> 2.000 m2): {luas_atas:,} baris")

# 3. Anomali Kamar Tidur & Kamar Mandi
kamar_nol = (df_raw['jumlah_kamar'] == 0).sum()
kamar_ekstrem = (df_raw['jumlah_kamar'] > 20).sum()
km_ekstrem = (df_raw['jumlah_kamar_mandi'] > 20).sum()
print(f"3. Properti Tanpa Kamar Tidur (kamar == 0): {kamar_nol:,} baris (Potensi Komersial)")
print(f"   Anomali Kamar Tidur (> 20 kamar): {kamar_ekstrem:,} baris")
print(f"   Anomali Kamar Mandi (> 20 kamar mandi): {km_ekstrem:,} baris")

# 4. Anomali Usia Bangunan
anomali_usia = df_raw[(df_raw['usia_bangunan'] > 100) & (df_raw['usia_bangunan'] < 1900)]['usia_bangunan']
print(f"4. Nilai Placeholder Scraper pada Usia Bangunan (77..1899): {len(anomali_usia):,} baris")
if len(anomali_usia) > 0:
    print(f"   Contoh nilai anomali: {sorted(anomali_usia.unique()[:10])}")

### 1.6 Visualisasi EDA Awal: Profil Missing Values & Deteksi Outlier Ekstrem
Menampilkan grafik batang persentase data hilang dan boxplot distribusi harga serta luas tanah dalam skala logaritmik.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Plot 1: Profil Nilai Hilang
missing_plot_data = missing_df[missing_df['Jumlah_Missing'] > 0]
sns.barplot(
    x=missing_plot_data['Persentase_Missing (%)'],
    y=missing_plot_data.index,
    palette='Reds_r',
    ax=axes[0]
)
axes[0].set_title('Profil Persentase Nilai Hilang (Missing Values)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Persentase Data Hilang (%)', fontsize=10)
axes[0].set_ylabel('Nama Fitur', fontsize=10)
for p in axes[0].patches:
    width = p.get_width()
    axes[0].annotate(f'{width:.1f}%', (width + 1, p.get_y() + p.get_height() / 2.),
                     ha='left', va='center', fontsize=9)

# Plot 2: Boxplot Outlier Skala Log
plot_data = df_raw[['harga', 'luas_m2']].dropna()
sns.boxplot(data=plot_data, orient='h', palette='Blues', ax=axes[1])
axes[1].set_xscale('log')
axes[1].set_title('Deteksi Outlier Ekstrem Fitur Utama (Skala Logaritmik)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Nilai Fitur (Skala Logaritmik)', fontsize=10)

plt.tight_layout()
plt.show()

---
## Bagian 2 — Pembersihan Data Terverifikasi Empiris (Adaptasi Praktikum 2)

Pada bagian ini, kita menerapkan pembersihan data terstruktur yang telah dirumuskan dan dibuktikan secara empiris pada `DECISION_LOG.md`. Pembersihan dilakukan secara bertahap dan transparan:

1. **Deduplikasi Baris Persis (`drop_duplicates()`):** Menghilangkan baris identik guna mencegah distorsi kepadatan (*density distortion*) dan kebocoran data pelatihan ke pengujian (*train-test data leakage*).
2. **Filter Outlier Domain:** Membuang anomali scraping yang melanggar batas kewajaran properti Indonesia.
3. **Retensi Properti Komersial (`jumlah_kamar == 0`):** Pembuktian empiris bahwa properti tanpa kamar tidur adalah ruko/komersial bernilai ekonomi tinggi.
4. **Perbaikan Nilai `usia_bangunan` & Ekstraksi Fitur `kondisi`:** Mengonversi tahun kalender (1900–2026) menjadi usia relatif per tahun referensi 2026, membuang placeholder scraper, dan menurunkan fitur kategori `kondisi`.
5. **Normalisasi String Lokasi (`lokasi_bersih`):** Melakukan standardisasi penulisan nama kota/kabupaten.
6. **Pembentukan Target Kuantil Seimbang:** Membentuk target 3 kelas (`kelas_harga`) dan target 2 kelas (`kelas_harga_biner`) berbasis kuantil empiris bebas subjektivitas.

Setiap tahap dicatat jumlah barisnya untuk menghasilkan **Tabel Audit Jejak Pembersihan (*Waterfall Log*)** yang tepat menyisakan **56.170 baris**.

In [ ]:
# ==============================================================================
# TAHAP 1: DEDUPLIKASI BARIS PERSIS
# ==============================================================================
df_step1 = df_raw.drop_duplicates().copy()
n_dropped_dup = len(df_raw) - len(df_step1)
pct_dropped_dup = (n_dropped_dup / len(df_raw)) * 100

print("=" * 70)
print("TAHAP 1: PENGHAPUSAN BARIS DUPLIKAT PERSIS")
print("=" * 70)
print(f"Baris Awal        : {len(df_raw):,} baris")
print(f"Baris Duplikat    : {n_dropped_dup:,} baris ({pct_dropped_dup:.2f}%)")
print(f"Baris Tersisa     : {len(df_step1):,} baris")
print("\nJustifikasi: Mencegah train-test leakage dan distorsi kepadatan pada clustering.")

### 2.2 Filter Outlier Domain Fisik & Finansial
Menerapkan filter gabungan berbasis batas kewajaran domain properti Indonesia:
- $10 \le \text{luas\_m2} \le 2.000$ m²
- $\text{luas\_bangunan}$ bernilai `NaN` atau $10 \le \text{luas\_bangunan} \le 2.000$ m²
- Rp 50.000.000 $\le \text{harga} \le$ Rp 100.000.000.000 (Rp 50 Jt – Rp 100 Miliar)
- $\text{jumlah\_kamar}$ bernilai `NaN` atau $\le 20$
- $\text{jumlah\_kamar\_mandi}$ bernilai `NaN` atau $\le 20$

In [ ]:
# ==============================================================================
# TAHAP 2: PENYARINGAN OUTLIER DOMAIN PROPERTI
# ==============================================================================
cond_luas_m2 = df_step1['luas_m2'].between(10, 2000)
cond_luas_bangunan = df_step1['luas_bangunan'].isna() | df_step1['luas_bangunan'].between(10, 2000)
cond_harga = df_step1['harga'].between(50_000_000, 100_000_000_000)
cond_kamar = df_step1['jumlah_kamar'].isna() | (df_step1['jumlah_kamar'] <= 20)
cond_km = df_step1['jumlah_kamar_mandi'].isna() | (df_step1['jumlah_kamar_mandi'] <= 20)

valid_mask = cond_luas_m2 & cond_luas_bangunan & cond_harga & cond_kamar & cond_km
df_step2 = df_step1[valid_mask].copy()

n_dropped_outliers = len(df_step1) - len(df_step2)
pct_dropped_outliers = (n_dropped_outliers / len(df_raw)) * 100

print("=" * 70)
print("TAHAP 2: HASIL PENYARINGAN OUTLIER DOMAIN")
print("=" * 70)
print(f"Baris Sebelum Filter : {len(df_step1):,} baris")
print(f"Baris Outlier Dibuang: {n_dropped_outliers:,} baris ({pct_dropped_outliers:.2f}%)")
print(f"Baris Tersisa        : {len(df_step2):,} baris")
assert len(df_step2) == 56170, f"Error: Jumlah baris harus tepat 56.170, didapatkan {len(df_step2)}"
print("Verifikasi Konsistensi Kontrak DECISION_LOG.md: TEPAT 56.170 BARIS BERHASIL!")

In [ ]:
# ==============================================================================
# PEMBUKTIAN EMPIRIS RETENSI PROPERTI KOMERSIAL (JUMLAH_KAMAR == 0)
# ==============================================================================
komersil_mask = df_step2['jumlah_kamar'] == 0
hunian_mask = df_step2['jumlah_kamar'] >= 1

n_komersil = komersil_mask.sum()
pct_komersil = (n_komersil / len(df_step2)) * 100
med_harga_komersil = df_step2.loc[komersil_mask, 'harga'].median()
med_harga_hunian = df_step2.loc[hunian_mask, 'harga'].median()

print("=" * 75)
print("BUKTI EMPIRIS RETENSI PROPERTI KOMERSIAL (JUMLAH_KAMAR == 0)")
print("=" * 75)
print(f"Jumlah sampel komersial (kamar == 0) : {n_komersil:,} baris ({pct_komersil:.2f}% dari data)")
print(f"Median Harga Komersial (kamar == 0)   : Rp {med_harga_komersil:,.2f} (~Rp 1,34 Miliar)")
print(f"Median Harga Hunian (kamar >= 1)      : Rp {med_harga_hunian:,.2f} (~Rp 750 Juta)")
print(f"Rasio Harga Komersial vs Hunian       : {med_harga_komersil / med_harga_hunian:.2f}x lebih tinggi")
print("-" * 75)
print("Kesimpulan Bisnis:")
print("Listing dengan kamar == 0 memiliki median harga Rp 1,34 Miliar (1,78x lebih mahal dari rumah biasa).")
print("Ini membuktikan bahwa kamar == 0 adalah properti komersial bernilai tinggi (ruko/tanah/kantor),")
print("BUKAN data rusak/cacat. Data ini sah dipertahankan untuk fitur 'is_komersil' di JS03.")

### 2.3 Perbaikan Nilai `usia_bangunan` & Penurunan Fitur `kondisi`
Sesuai audit EDA awal, nilai pada kolom `usia_bangunan` mengalami pencampuran format:
1. Sebagian entri berupa **tahun kalender pembangunan** (misal `2018`, `2020`).
2. Sebagian entri berupa **usia relatif** (misal `2`, `5` tahun).
3. Sebagian entri berupa **placeholder scraper cacat** (misal `1111`, `125`, `220`).

Dengan tahun referensi proyek **`TAHUN_INI = 2026`**, kita melakukan konversi:
- Nilai $1900 \le x \le 2026 \rightarrow 2026 - x$.
- Nilai di luar batas $0 \le x \le 100$ diubah menjadi `NaN`.
- Diturunkan fitur kategorikal `kondisi`:
  - `Baru` jika usia $\le 3$ tahun.
  - `Bekas` jika usia $> 3$ tahun.
  - `Tidak diketahui` jika usia tidak terisi (`NaN`).

In [ ]:
# ==============================================================================
# TAHAP 3: PERBAIKAN USIA BANGUNAN & PENURUNAN FITUR 'KONDISI'
# ==============================================================================
TAHUN_INI = 2026

def normalize_age(val):
    if pd.isna(val):
        return np.nan
    try:
        val = float(val)
    except (ValueError, TypeError):
        return np.nan
    
    # Konversi tahun kalender ke umur
    if 1900 <= val <= TAHUN_INI:
        val = TAHUN_INI - val
    
    # Filter umur valid realistis (0 - 100 tahun)
    if 0 <= val <= 100:
        return val
    else:
        return np.nan

df_step3 = df_step2.copy()
df_step3['usia_bangunan'] = df_step3['usia_bangunan'].apply(normalize_age)

# Penurunan fitur kategorikal kondisi
def derive_condition(age):
    if pd.isna(age):
        return 'Tidak diketahui'
    elif age <= 3:
        return 'Baru'
    else:
        return 'Bekas'

df_step3['kondisi'] = df_step3['usia_bangunan'].apply(derive_condition)

print("=" * 70)
print("PERBAIKAN USIA BANGUNAN & PENURUNAN FITUR 'KONDISI'")
print("=" * 70)
print(f"Missing count usia_bangunan : {df_step3['usia_bangunan'].isna().sum():,} baris ({df_step3['usia_bangunan'].isna().mean()*100:.2f}%)")
print("\nDistribusi Frekuensi Fitur Turunan 'kondisi':")
print(df_step3['kondisi'].value_counts())

### 2.4 Normalisasi Lokasi & Pembentukan Target Kuantil Seimbang
1. **Normalisasi `lokasi_bersih`:** Menghilangkan spasi berlebih dan menyeragamkan kapitalisasi kata (`strip().title()`).
2. **Target 3 Kelas (`kelas_harga`):** Dibagi menjadi 3 tertil yang terdistribusi seimbang sempurna secara statistik (33,33% / 33,34% / 33,33%):
   - `Murah`: $< q_1$ (< Rp 483.819.314,00)
   - `Sedang`: $q_1 \le \text{harga} \le q_2$ (Rp 483,82 Jt – Rp 1,235 Miliar)
   - `Mahal`: $> q_2$ (> Rp 1.235.000.000,00)
3. **Target 2 Kelas (`kelas_harga_biner`):** Dibagi berdasarkan nilai median (Rp 750.000.000,00):
   - `Murah`: $\le \text{Median}$ (50,68%)
   - `Mahal`: $> \text{Median}$ (49,32%)

In [ ]:
# ==============================================================================
# TAHAP 4 & 5: NORMALISASI LOKASI & PEMBENTUKAN TARGET KUANTIL
# ==============================================================================
df_step4 = df_step3.copy()

# Normalisasi string lokasi
df_step4['lokasi_bersih'] = df_step4['lokasi'].astype(str).str.strip().str.title()
n_lokasi_unik = df_step4['lokasi_bersih'].nunique()
print(f"[TAHAP 4] Normalisasi Lokasi: Menghasilkan {n_lokasi_unik} lokasi bersih unik.")

# Pembentukan Target Kuantil
q1 = df_step4['harga'].quantile(1/3)
q2 = df_step4['harga'].quantile(2/3)
med = df_step4['harga'].median()

def categorize_price_3(val):
    if val < q1:
        return 'Murah'
    elif val <= q2:
        return 'Sedang'
    else:
        return 'Mahal'

def categorize_price_2(val):
    if val <= med:
        return 'Murah'
    else:
        return 'Mahal'

df_step4['kelas_harga'] = df_step4['harga'].apply(categorize_price_3)
df_step4['kelas_harga_biner'] = df_step4['harga'].apply(categorize_price_2)

print("\n" + "=" * 70)
print("PEMBENTUKAN TARGET KLASIFIKASI BERBASIS KUANTIL SEIMBANG")
print("=" * 70)
print(f"Ambang Batas Tertil 1 (q1) : Rp {q1:,.2f}")
print(f"Ambang Batas Tertil 2 (q2) : Rp {q2:,.2f}")
print(f"Nilai Median (50%)         : Rp {med:,.2f}")

print("\nDistribusi Target 3 Kelas ('kelas_harga'):")
t3_dist = pd.DataFrame({
    'Jumlah Baris': df_step4['kelas_harga'].value_counts(),
    'Persentase (%)': (df_step4['kelas_harga'].value_counts(normalize=True) * 100).round(2)
})
display(t3_dist)

print("\nDistribusi Target 2 Kelas ('kelas_harga_biner'):")
t2_dist = pd.DataFrame({
    'Jumlah Baris': df_step4['kelas_harga_biner'].value_counts(),
    'Persentase (%)': (df_step4['kelas_harga_biner'].value_counts(normalize=True) * 100).round(2)
})
display(t2_dist)

In [ ]:
# ==============================================================================
# TABEL RINGKASAN JEJAK PEMBERSIHAN DATA (WATERFALL CLEANING LOG)
# ==============================================================================
audit_records = [
    {'Tahap': '1. Dataset Mentah Scraped', 'Baris': len(df_raw), 'Baris_Dibuang': 0, 'Persen_Sisa': 100.0},
    {'Tahap': '2. Deduplikasi Persis', 'Baris': len(df_step1), 'Baris_Dibuang': n_dropped_dup, 'Persen_Sisa': (len(df_step1)/len(df_raw))*100},
    {'Tahap': '3. Filter Outlier Domain & Validasi Komersial', 'Baris': len(df_step2), 'Baris_Dibuang': n_dropped_outliers, 'Persen_Sisa': (len(df_step2)/len(df_raw))*100},
    {'Tahap': '4. Normalisasi Usia & Rekayasa Target', 'Baris': len(df_step4), 'Baris_Dibuang': 0, 'Persen_Sisa': (len(df_step4)/len(df_raw))*100}
]
audit_df = pd.DataFrame(audit_records)
print("=" * 75)
print("TABEL AUDIT JEJAK PEMBERSIHAN DATA (WATERFALL CLEANING LOG)")
print("=" * 75)
display(audit_df.style.format({'Baris': '{:,}', 'Baris_Dibuang': '{:,}', 'Persen_Sisa': '{:.2f}%'}))

df_cleaned = df_step4.copy()
print(f"\nStatus Akhir Data Bersih: {df_cleaned.shape[0]:,} baris, {df_cleaned.shape[1]} kolom.")

---
## Bagian 3 — Demonstrasi Imputasi, Encoding, & Standardisasi (Adaptasi Praktikum 3 & Tugas)

> ### ⚠️ PENTING: PRINSIP ANTI-DATA LEAKAGE SHOWCASE
> Terdapat perbedaan metodologis yang sangat mendasar antara **demonstrasi eksploratif di `JS02`** versus **implementasi produksi pemodelan di `JS04–JS15`**:
> 
> 1. **Aturan Penyimpanan CSV Bersih (`properti_multi_kota_clean.csv`):**
>    - Kolom `luas_bangunan` (missing 2,73%), `jumlah_kamar` (missing 0,61%), dan `jumlah_kamar_mandi` (missing 0,63%) **SENGAJA DISIMPAN MEMILIKI NILAI `NaN` (TIDAK DIIMPUTASI & TIDAK DISTANDARISASI DI FILE CSV)**.
>    - *Landasan Teori:* Jika kita mengimputasi median dan melakukan standardisasi pada seluruh dataset sebelum pemisahan train/test, nilai rata-rata dan variansi data uji telah mengontaminasi data latih (**Data Leakage**). Akibatnya, model akan menghasilkan skor performa yang overoptimistik secara palsu.
>    - Pada tahap pemodelan (`JS04–JS15`), seluruh imputasi median, encoding kategori, dan scaling numerik wajib dimasukkan ke dalam `Pipeline` / `ColumnTransformer` scikit-learn **SETELAH** fungsi `train_test_split` dipanggil.
> 
> 2. **Tujuan Demonstrasi di Bagian 3 Ini:**
>    - Mendemonstrasikan teknik teknis `SimpleImputer`, perbandingan strategi encoding kategorikal, dan `StandardScaler` pada salinan data eksploratif (**`df_demo`**) agar mahasiswa memahami cara kerja matematisnya sesuai kurikulum `JS02`.

In [ ]:
# ==============================================================================
# DEMONSTRASI 1: IMPUTASI MEDIAN PADA FITUR NUMERIK SKEWED
# ==============================================================================
df_demo = df_cleaned.copy()
num_impute_cols = ['luas_bangunan', 'jumlah_kamar', 'jumlah_kamar_mandi']

print("=" * 70)
print("DEMONSTRASI 1: IMPUTASI MEDIAN PADA FITUR NUMERIK SKEWED")
print("=" * 70)
print("Jumlah Missing Values Sebelum Imputasi:")
print(df_demo[num_impute_cols].isnull().sum())

# Hitung median dari data
median_values = {col: df_demo[col].median() for col in num_impute_cols}
print("\nNilai Median Terhitung:")
for col, val in median_values.items():
    print(f" - {col}: {val}")

# Imputasi dengan median
for col in num_impute_cols:
    df_demo[col] = df_demo[col].fillna(median_values[col])

print("\nJumlah Missing Values Setelah Imputasi:")
print(df_demo[num_impute_cols].isnull().sum())
print("\nJustifikasi: Median dipilih alih-alih mean karena distribusi fitur properti")
print("memiliki kemiringan positif (right-skewed), sehingga mean rentan terdistorsi outlier.")

In [ ]:
# ==============================================================================
# DEMONSTRASI 2: KOMPARASI STRATEGI ENCODING KOLOM KATEGORIKAL TINGGI
# ==============================================================================
print("=" * 75)
print("DEMONSTRASI 2: KOMPARASI STRATEGI ENCODING KOLOM KATEGORIKAL TINGGI")
print("=" * 75)

# Strategi A: Frequency Encoding
freq_map = df_demo['lokasi_bersih'].value_counts(normalize=True).to_dict()
df_demo['lokasi_freq'] = df_demo['lokasi_bersih'].map(freq_map)

# Strategi B: One-Hot Encoding Top-15 + 'Lainnya'
top15_cities = df_demo['lokasi_bersih'].value_counts().nlargest(15).index.tolist()
df_demo['lokasi_top15'] = df_demo['lokasi_bersih'].apply(lambda x: x if x in top15_cities else 'Lainnya')

top15_coverage = df_demo['lokasi_bersih'].isin(top15_cities).mean() * 100

print(f"Total Nilai Unik Lokasi Bersih          : {df_demo['lokasi_bersih'].nunique()} kategori")
print("Dimensi Hasil One-Hot Encoding Mentah   : 588 kolom biner (Curse of Dimensionality!)")
print(f"Cakupan Data Top-15 Kota                : {top15_coverage:.2f}% dari seluruh data")
print("Dimensi Hasil Top-15 + 'Lainnya'        : 16 kolom biner")
print("Dimensi Hasil Frequency Encoding        : 1 kolom kontinu")

print("\nTop-15 Kota Terbanyak dan Proporsinya:")
display(pd.DataFrame({
    'Jumlah_Listing': df_demo['lokasi_bersih'].value_counts().head(15),
    'Persentase (%)': (df_demo['lokasi_bersih'].value_counts(normalize=True).head(15) * 100).round(2)
}))

In [ ]:
# ==============================================================================
# DEMONSTRASI 3 & 4: STANDARDISASI STANDARDSCALER & TRANSFORMASI LOG TARGET
# ==============================================================================
scaler = StandardScaler()
scale_cols = ['luas_m2', 'luas_bangunan', 'jumlah_kamar', 'jumlah_kamar_mandi']

df_demo_scaled = df_demo.copy()
df_demo_scaled[[c + '_std' for c in scale_cols]] = scaler.fit_transform(df_demo[scale_cols])

print("=" * 75)
print("DEMONSTRASI 3: STANDARDISASI STANDARDSCALER (MEAN ~ 0, STD ~ 1)")
print("=" * 75)
scaled_stats = df_demo_scaled[[c + '_std' for c in scale_cols]].agg(['mean', 'std']).T
display(scaled_stats.style.format('{:.4f}'))

# Transformasi Logaritmik Target Regresi
df_demo['log1p_harga'] = np.log1p(df_demo['harga'])

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Plot Distribusi Harga Asli
sns.histplot(df_demo['harga'], bins=50, kde=True, color='crimson', ax=axes[0])
axes[0].set_title('Distribusi Harga Asli (Rupiah) — Heavy Right-Skewed', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Harga (Rupiah)', fontsize=10)
axes[0].set_ylabel('Frekuensi', fontsize=10)

# Plot Distribusi log1p(Harga)
sns.histplot(df_demo['log1p_harga'], bins=50, kde=True, color='royalblue', ax=axes[1])
axes[1].set_title('Distribusi log(1 + Harga) — Mendekati Distribusi Normal', fontsize=12, fontweight='bold')
axes[1].set_xlabel('log1p(Harga)', fontsize=10)
axes[1].set_ylabel('Frekuensi', fontsize=10)

plt.tight_layout()
plt.show()

print("\nJustifikasi Teoretis:")
print("Transformasi log1p(harga) menstabilkan variansi residual (homoskedastisitas) dan")
print("membuat metrik jarak Euclidean pada KNN/Clustering menjadi bermakna.")

---
## Bagian 4 — Eksplorasi Pasca-Praproses & Korelasi Multivariat

Pada bagian ini, kita melakukan analisis korelasi dan eksplorasi multivariat untuk memverifikasi hubungan antar variabel yang telah dibersihkan:
1. **Visualisasi 3 Panel:**
   - Panel 1: Distribusi harga properti (skala logaritmik) berdasarkan kategori `kondisi` bangunan (`Baru`, `Bekas`, `Tidak diketahui`).
   - Panel 2: Distribusi luas tanah (`luas_m2`) terhadap target `kelas_harga`.
   - Panel 3: Pola hubungan non-linear luas tanah versus harga properti (3.000 titik sampel acak).
2. **Matriks Korelasi Pearson:** Menilai kekuatan korelasi linier fitur numerik terhadap harga asli dan target logaritmik `log1p(harga)`.

In [ ]:
# ==============================================================================
# VISUALISASI 3 PANEL PASCA-PRAPROSES
# ==============================================================================
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Panel 1: Boxplot harga (log scale) per kondisi
sns.boxplot(x='kondisi', y='harga', data=df_demo, palette='Set2', ax=axes[0],
            order=['Baru', 'Bekas', 'Tidak diketahui'])
axes[0].set_yscale('log')
axes[0].set_title('Distribusi Harga Properti per Kondisi Bangunan', fontsize=11, fontweight='bold')
axes[0].set_xlabel('Kondisi Bangunan', fontsize=10)
axes[0].set_ylabel('Harga Properti (Skala Log)', fontsize=10)

# Panel 2: Boxplot luas_m2 per kelas_harga
sns.boxplot(x='kelas_harga', y='luas_m2', data=df_demo, palette='Set1', ax=axes[1],
            order=['Murah', 'Sedang', 'Mahal'])
axes[1].set_title('Distribusi Luas Tanah per Kelas Harga', fontsize=11, fontweight='bold')
axes[1].set_xlabel('Kelas Harga', fontsize=10)
axes[1].set_ylabel('Luas Tanah (m2)', fontsize=10)

# Panel 3: Scatter plot luas_m2 vs harga (log scale, sample 3000 titik)
sample_demo = df_demo.sample(n=3000, random_state=42)
sns.scatterplot(
    data=sample_demo,
    x='luas_m2',
    y='harga',
    hue='kelas_harga',
    hue_order=['Murah', 'Sedang', 'Mahal'],
    palette='viridis',
    alpha=0.6,
    s=25,
    ax=axes[2]
)
axes[2].set_yscale('log')
axes[2].set_title('Hubungan Luas Tanah vs Harga Properti (3.000 Sampel)', fontsize=11, fontweight='bold')
axes[2].set_xlabel('Luas Tanah (m2)', fontsize=10)
axes[2].set_ylabel('Harga (Skala Log)', fontsize=10)

plt.tight_layout()
plt.show()

In [ ]:
# ==============================================================================
# HEATMAP MATRIKS KORELASI FITUR NUMERIK
# ==============================================================================
corr_cols = ['luas_m2', 'luas_bangunan', 'jumlah_kamar', 'jumlah_kamar_mandi', 'usia_bangunan', 'harga', 'log1p_harga']
corr_matrix = df_demo[corr_cols].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(
    corr_matrix,
    annot=True,
    fmt='.3f',
    cmap='coolwarm',
    vmin=-1.0,
    vmax=1.0,
    linewidths=0.5
)
plt.title('Matriks Korelasi Pearson Fitur Numerik terhadap Harga & log1p(Harga)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print("=" * 65)
print("KORELASI FITUR NUMERIK TERHADAP TARGET LOGARITMIK (log1p_harga)")
print("=" * 65)
print(corr_matrix['log1p_harga'].sort_values(ascending=False))

---
## Kesimpulan & Panduan Interpretasi EDA Jobsheet 02

Berdasarkan seluruh rangkaian eksplorasi dan pra-pengolahan data mentah pada jobsheet ini, diperoleh kesimpulan empiris sebagai berikut:

1. **Hasil Pembersihan Empiris Data Mentah:**
   - Dataset mentah sebanyak **64.650 baris** berhasil dibersihkan menjadi **56.170 baris data valid** (membuang 7.156 baris duplikat persis dan 1.324 baris outlier ekstrem fisik/finansial).
   - Seluruh baris yang dipertahankan konsisten 100% dengan standar kontrak teknis `DECISION_LOG.md`.

2. **Bukti Validitas Properti Komersial (`jumlah_kamar == 0`):**
   - Sampel properti dengan 0 kamar tidur sebanyak **2.252 baris (4,01%)** terbukti memiliki median harga **Rp 1,34 Miliar** (1,78x lebih tinggi dari hunian biasa). Ini membuktikan bahwa baris tersebut merupakan properti komersial bernilai tinggi (ruko/kantor/tanah) dan bukan data rusak.

3. **Integritas Target Kuantil Seimbang:**
   - Pembagian target 3 kelas (`kelas_harga`: `Murah` < Rp 483,82 jt, `Sedang` Rp 483,82 jt–Rp 1,235 M, `Mahal` > Rp 1,235 M) menghasilkan proporsi seimbang sempurna (33,33% / 33,34% / 33,33%), bebas dari subjektivitas ambang batas dan mencegah distorsi *class imbalance*.

4. **Korelasi Terhadap Target:**
   - Korelasi fitur terhadap target harga logaritmik menunjukkan bahwa `luas_bangunan` ($r \approx 0,65$) dan `luas_m2` ($r \approx 0,55$) merupakan prediktor linear terkuat, disusul oleh fasilitas kamar tidur dan kamar mandi.
   - Variabel `usia_bangunan` memiliki missing rate tinggi (92,29%), sehingga diturunkan menjadi fitur kategorikal penunjang `kondisi` alih-alih fitur numerik primer.

5. **Prinsip Anti-Data Leakage untuk Jobsheet Pemodelan (`JS04–JS15`):**
   - File dataset bersih `Dataset/properti_multi_kota_clean.csv` sengaja menyimpan nilai `NaN` tanpa imputasi. Seluruh proses imputasi, encoding lokasi (Top-15 + 'Lainnya'), dan scaling numerik wajib dilakukan di dalam `Pipeline` scikit-learn setelah `train_test_split` pada jobsheet pemodelan berikutnya.